In [ ]:
import sys
import os

import pandas as pd
import json
from datetime import datetime, timedelta
import logging


def main():
    setup_logging()
    logger = logging.getLogger(__name__)
    
    logger.info("Starting data collection...")

    oil_collector = OilPriceCollector()
    flight_scraper = FlightRadarScraper()
    base_monitor = BaseMonitor()
    
    try:
        Path("data/raw/flight_data").mkdir(parents=True, exist_ok=True)
        Path("data/raw/oil_prices").mkdir(parents=True, exist_ok=True)

        logger.info("Collecting oil price data...")
        current_prices = oil_collector.fetch_current_prices()
        historical_oil = oil_collector.fetch_historical_data(days=90)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        oil_file = f"data/raw/oil_prices/oil_prices_{timestamp}.csv"
        historical_oil.to_csv(oil_file, index=False)
        
        with open(f"data/raw/oil_prices/current_prices_{timestamp}.json", 'w') as f:
            json.dump(current_prices, f)
        
        logger.info(f"Oil data saved to {oil_file}")

        logger.info("Collecting flight data...")
        flights = flight_scraper.get_middle_east_flights()
        
        if flights:
            flight_df = pd.DataFrame(flights)

            flight_df['is_military'] = flight_df['callsign'].apply(
                base_monitor.is_military_aircraft
            )

            base_info = []
            for _, flight in flight_df.iterrows():
                is_near, base_name = base_monitor.is_near_base(
                    flight['latitude'], flight['longitude']
                )
                base_info.append({
                    'is_near_base': is_near,
                    'base_name': base_name if is_near else 'None'
                })
            
            base_df = pd.DataFrame(base_info)
            flight_df = pd.concat([flight_df, base_df], axis=1)

            flight_file = f"data/raw/flight_data/flights_{timestamp}.csv"
            flight_df.to_csv(flight_file, index=False)

            activity_summary = base_monitor.categorize_activity(flights)
            with open(f"data/raw/flight_data/activity_summary_{timestamp}.json", 'w') as f:
                json.dump(activity_summary, f)
            
            logger.info(f"Flight data saved to {flight_file}")
            logger.info(f"Military flights detected: {activity_summary['total_military_flights']}")
            
        else:
            logger.warning("No flight data collected")
        
    except Exception as e:
        logger.error(f"Error during data collection: {e}")
        return 1
    
    finally:
        flight_scraper.close_driver()
    
    logger.info("Data collection completed successfully")
    return 0

if __name__ == "__main__":
    exit(main())

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pandas as pd
import numpy as np
from pathlib import Path
import logging
from datetime import datetime

from src.ml_models.feature_engineering import FeatureEngineer
from src.ml_models.predictor_model import OilPricePredictor
from src.ml_models.model_evaluator import ModelEvaluator
from src.utils.logging_config import setup_logging

def load_data():
    oil_files = List(Path("data/raw/oil_prices").glob("oil_prices_*.csv"))
    if not oil_files:
        raise FileNotFoundError("No oil price data found")
    
    latest_oil_file = max(oil_files, key=os.path.getctime)
    oil_data = pd.read_csv(latest_oil_file)
    oil_data['Timestamp'] = pd.to_datetime(oil_data['Timestamp'])

    flight_files = List(Path("data/raw/flight_data")).glob("flights_*.csv")
    if not flight_files:
        raise FileNotFoundError("No flight data found")
    
    flight_data_list = []
    for flight in flight_files:
        df = pd.read_csv(file)
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
        flight_data_list.append(df)

    flight_data = pd.concat(flight_data_list, ignore_index=True)

    return oil_data, flight_data

def main():
    setup_logging()
    logger = logging.getLogger(__name__)

    logger.info("Starting Model training...")

    try:
        logger.info("Loading data...")
        oil_data, flight_data = load_data()

        feature_engineer = FeatureEngineer()
     
        logger.info("Engineering features...")
        flight_features = feature_engineer.create_flight_features(flight_data)
        oil_features = feature_engineer.create_oil_features(oil_data)

        combined_features = feature_engineer.combine_features(
            flight_features, oil_features
        )
        
        logger.info(f"Generated {len(combined_features.columns)} features")
        logger.info(f"Dataset shape: {combined_features.shape}")

        predictor = OilPricePredictor()
        X, y = predictor.prepare_data(combined_features, target_column='bz_price')
        
        logger.info(f"Training data shape: X={X.shape}, y={y.shape}")

        logger.info("Training models...")
        results = predictor.train_models(X, y)

        evaluator = ModelEvaluator()
        y_pred = predictor.predict(X)
        
        metrics = evaluator.calculate_metrics(y, y_pred)
        evaluator.print_evaluation_report(metrics)

        logger.info("Generating evaluation plots...")
        evaluator.plot_predictions(y, y_pred, "Oil Price Prediction Results")
        
        if predictor.feature_importance is not None:
            evaluator.plot_feature_importance(
                predictor.feature_importance, 
                list(combined_features.columns)
            )

        Path("data/models").mkdir(parents=True, exist_ok=True)
        model_path = f"data/models/oil_predictor_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
        predictor.save_model(model_path)

        results_df = pd.DataFrame(results).T
        results_path = f"data/models/training_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        results_df.to_csv(results_path)
        
        logger.info(f"Model saved to {model_path}")
        logger.info(f"Training results saved to {results_path}")
        
    except Exception as e:
        logger.error(f"Error during model training: {e}")
        return 1
    
    logger.info("Model training completed successfully")
    return 0

if __name__ == "__main__":
    exit(main())

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pandas as pd
import numpy as np
from pathlib import Path
import logging
from datetime import datetime

from src.ml_models.predictor_model import OilPricePredictor
from src.ml_models.feature_engineering import FeatureEngineer
from src.data_collection.oil_price_collector import OilPriceCollector
from src.data_collection.flightradar_scraper import FlightRadarScraper
from src.data_collection.base_monitor import BaseMonitor
from src.utils.logging_config import setup_logging

def load_latest_model():
    model_files = list(Path("data/models").glob("oil_predictor_*.pkl"))
    if not model_files:
        raise FileNotFoundError("No trained models found")
    
    latest_model = max(model_files, key=os.path.getctime)
    
    predictor = OilPricePredictor()
    predictor.load_model(latest_model)
    
    return predictor

def collect_real_time_data():
    oil_collector = OilPriceCollector()
    flight_scraper = FlightRadarScraper()
    base_monitor = BaseMonitor()
    
    try:
        current_prices = oil_collector.fetch_current_prices()
        recent_oil = oil_collector.fetch_historical_data(days=7)
        
        flights = flight_scraper.get_middle_east_flights()
        
        if flights:
            flight_df = pd.DataFrame(flights)
            flight_df['is_military'] = flight_df['callsign'].apply(
                base_monitor.is_military_aircraft
            )

            base_info = []
            for _, flight in flight_df.iterrows():
                is_near, base_name = base_monitor.is_near_base(
                    flight['latitude'], flight['longitude']
                )
                base_info.append({
                    'is_near_base': is_near,
                    'base_name': base_name if is_near else 'None'
                })
            
            base_df = pd.DataFrame(base_info)
            flight_df = pd.concat([flight_df, base_df], axis=1)
            
            return recent_oil, flight_df, current_prices
        
        return recent_oil, pd.DataFrame(), current_prices
    
    finally:
        flight_scraper.close_driver()

def main():
    setup_logging()
    logger = logging.getLogger(__name__)
    
    logger.info("Starting oil price prediction...")
    
    try:
        logger.info("Loading trained model...")
        predictor = load_latest_model()
        
        logger.info("Collecting real-time data...")
        oil_data, flight_data, current_prices = collect_real_time_data()
        
        logger.info("Engineering features for prediction...")
        feature_engineer = FeatureEngineer()
        
        if not flight_data.empty:
            flight_features = feature_engineer.create_flight_features(flight_data)
        else:
            logger.warning("No current flight data available")
            flight_features = pd.DataFrame()
        
        oil_features = feature_engineer.create_oil_features(oil_data)

        if not flight_features.empty:
            combined_features = feature_engineer.combine_features(
                flight_features, oil_features
            )
        else:
            combined_features = oil_features
            logger.warning("Prediction based only on oil price features")

        logger.info("Making predictions...")

        latest_features = combined_features.iloc[-1:].values
        prediction = predictor.predict(latest_features)[0]

        logger.info("="*60)
        logger.info("OIL PRICE PREDICTION REPORT")
        logger.info("="*60)
        logger.info(f"Prediction Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.info(f"Current Brent Crude (BZ=F): ${current_prices.get('BZ=F', 'N/A')}")
        logger.info(f"Current WTI Crude (CL=F): ${current_prices.get('CL=F', 'N/A')}")
        logger.info(f"")
        logger.info(f"PREDICTED BRENT CRUDE PRICE (Next Period): ${prediction:.2f}")
        
        if not flight_data.empty:
            military_count = flight_data['is_military'].sum()
            total_flights = len(flight_data)
            logger.info(f"")
            logger.info(f"Current Military Activity:")
            logger.info(f"  Total Flights Monitored: {total_flights}")
            logger.info(f"  Military Flights: {military_count}")
            logger.info(f"  Military Flight Ratio: {military_count/total_flights:.2%}")
        
        logger.info("="*60)

        prediction_data = {
            'timestamp': datetime.now().isoformat(),
            'predicted_price': float(prediction),
            'current_bz_price': current_prices.get('BZ=F'),
            'current_cl_price': current_prices.get('CL=F'),
            'military_flights': int(flight_data['is_military'].sum()) if not flight_data.empty else 0,
            'total_flights': len(flight_data),
            'model_used': predictor.best_model_name
        }

        Path("data/predictions").mkdir(parents=True, exist_ok=True)
        prediction_file = f"data/predictions/prediction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
        import json
        with open(prediction_file, 'w') as f:
            json.dump(prediction_data, f, indent=2)
        
        logger.info(f"Prediction saved to {prediction_file}")
        
    except Exception as e:
        logger.error(f"Error during prediction: {e}")
        return 1
    
    logger.info("Prediction completed successfully")
    return 0

if __name__ == "__main__":
    exit(main())